In [2]:


!pip install graphviz
!pip install lark
from graphviz import Digraph
from lark import Lark

global_token_counter = 0
def get_next_token_number():
    global global_token_counter
    global_token_counter += 1
    return global_token_counter

def reset_token_counter():
    global global_token_counter
    global_token_counter = 0


def debug_chars(s):
    out = ''.join(f'{c}({ord(c):04X})' for c in s)
    print(out)
    return out


def preprocess_input(raw_input):
    raw_input = re.sub(r' {5,}', '\n', raw_input)
    debug_chars(raw_input)
    lines = [line.strip() for line in raw_input.splitlines() if line.strip()]

    wrapped_lines = []
    for line in lines:
        norm = normalize_line_operators(line)
        validate_operator_spacing(norm)
        norm = normalize_top_level_alternatives(norm)
        if not norm.startswith("START"):
            norm = "START " + norm
        if not norm.endswith("SINK"):
            norm = norm + " SINK"
        wrapped_lines.append(norm)
    return "\n".join(wrapped_lines)



file_path = "/content/drive/MyDrive/Lab Work/Bayesian_Graph_Metabolic_Paper/KEGG_Graph_Generation/new_updated_grammar.txt"
with open(file_path, "r") as f:
    grammar = f.read()
print(grammar)


parser = Lark(grammar, parser='lalr', start='file')


def tree_to_graphviz(tree, graph=None, parent_id=None, node_id=0):
    if graph is None:
        graph = Digraph()
    # Create a unique id for this node
    current_id = f"node{node_id}"
    label = tree.data if hasattr(tree, 'data') else str(tree)
    graph.node(current_id, label)

    # If there is a parent, create an edge
    if parent_id is not None:
        graph.edge(parent_id, current_id)

    node_id += 1

    # If it's a Tree node, iterate over children
    if hasattr(tree, 'children'):
        for child in tree.children:
            graph, node_id = tree_to_graphviz(child, graph, current_id, node_id)

    return graph, node_id



## LARK transformer to AST
## For processing matched rules and tokens

from lark import Transformer

class ASTNode:
    pass

class ProgramNode(ASTNode):
    def __init__(self, expressions):
        self.expressions = expressions
    def __repr__(self):
        return f"Program({self.expressions})"

class SequenceNode(ASTNode):
    def __init__(self, parts):
        self.parts = parts
    def __repr__(self):
        return f"Sequence({self.parts})"

class AlternativeNode(ASTNode):
    def __init__(self, alternatives):
        self.alternatives = alternatives
    def __repr__(self):
        return f"Alternative({self.alternatives})"

class MultiplyNode(ASTNode):
    def __init__(self, left, op, right):
        self.left = left
        self.op = op  # '+' or '-'
        self.right = right
    def __repr__(self):
        return f"Multiply({self.left} {self.op} {self.right})"

class StartNode(ASTNode):
    def __repr__(self):
        return "START"

class SinkNode(ASTNode):
    def __repr__(self):
        return "SINK"

class KoIDNode(ASTNode):
    def __init__(self, kid):
        self.kid = kid
    def __repr__(self):
        return f"{self.kid}"

class OptionalNode(ASTNode):
    def __init__(self, factor):
        self.factor = factor
    def __repr__(self):
        return f"Optional({self.factor})"

class ModuleNode(ASTNode):
    def __init__(self, module_id):
        self.module_id = module_id
    def __repr__(self):
        return f"Module({self.module_id})"



from lark import Transformer, Token


def _mk_sequence(items):
  parts = [x for x in items
    if not (isinstance(x, Token) and x.type == "SPACE")
    and not (isinstance(x, str) and not x.strip())]
  return parts[0] if len(parts) == 1 else SequenceNode(parts)


class ASTTransformer(Transformer):
    def file(self, items):
      return ProgramNode([x for x in items if not (isinstance(x, Token) and x.type == "NEWLINE")])


    def expr(self, items):
        # items = [START, top, SINK] (by grammar)
        return SequenceNode(items)

    # Outside-paren structure
    def top(self, items):
        # top: step (SPACE step)*  -> make a sequence of steps
        return _mk_sequence(items)

    def factor(self, items):
      return items[0]

    def unit(self, items):
      return items[0]  # lets ko_id/module handlers produce your nodes


    def step(self, items):
        # step: choice | chain
        return items[0]

    # Alternatives at depth-0 (outside parens)
    def choice(self, items):
        # choice: chain (COMMA chain)+
        branches = [x for x in items if not (isinstance(x, Token) and x.type == "COMMA")]
        return AlternativeNode(branches)


    def alt_list(self, items):
    # items may include COMMA tokens; keep only real branches
      branches = [x for x in items if not isinstance(x, Token)]
      if not branches:
          raise ValueError("alt_list has no branches")
      return AlternativeNode(branches)

    # +/- product over factors
    def chain(self, items):
        # chain: factor (PLUS factor | MINUS factor)*
        left = items[0]
        i = 1
        while i < len(items):
            tok = items[i]
            if isinstance(tok, Token) and tok.type in ("PLUS", "MINUS"):
                op = '+' if tok.type == "PLUS" else '-'
                right = items[i+1]
                i += 2
            else:
                raise ValueError(f"Unexpected token in chain at index {i}: {items[i]!r}")
            # '+' concatenates; '-' means optional right (your semantics)
            if op == '+':
                left = MultiplyNode(left, '+', right)
            else:
                left = MultiplyNode(left, '-', OptionalNode(right))
        return left

    # Groups / inside-paren structure
    def group(self, items):
        # group: LPAREN inner RPAREN  to drop punctuation, keep inner AST
        inner = [x for x in items if not isinstance(x, Token)]
        if not inner:
            raise ValueError("Empty parenthesized group")
        return inner[0]

    def inner(self, items):
        # inner: seq (COMMA seq)*  to alternatives of seq
        branches = [x for x in items if not (isinstance(x, Token) and x.type == "COMMA")]
        return branches[0] if len(branches) == 1 else AlternativeNode(branches)

    def seq(self, items):
        # seq: chain (SPACE chain)*  to sequence
        return _mk_sequence(items)

    # Terminals (same as before)
    def start_node(self, _):
        return StartNode()

    def sink_node(self, _):
        return SinkNode()

    def sequence(self, items):
      return _mk_sequence(items)

    def ko_id(self, items):
        token = items[0]
        value = token.value if hasattr(token, "value") else str(token)
        unique_number = get_next_token_number()
        if value.startswith('-'):
            ko_value = value[1:]
            return OptionalNode(KoIDNode(f"{ko_value}_{unique_number}"))
        else:
            return KoIDNode(f"{value}_{unique_number}")

    def module(self, items):
      token = items[0]
      return ModuleNode(token.value if isinstance(token, Token) else str(token))


import re
_re_space_before_plus = re.compile(r'\s+\+')
_re_space_after_plus  = re.compile(r'\+(?=\s+(?:K\d{5}|M\d{5}|\())')
_re_space_after_minus = re.compile(r'-(?=\s+(?:K\d{5}|M\d{5}|\())')
_re_dangling_minus = re.compile(r'-(?=\s*(?:SINK|$))')

def validate_operator_spacing(s: str):
    s = s or ""
    m = _re_space_before_plus.search(s)
    if m:
        i = m.start()
        raise ValueError(f"Space before '+' is not allowed near: …{s[max(0,i-6):i+8]}…")
    m = _re_space_after_plus.search(s)
    if m:
        i = m.start()
        raise ValueError(f"Space after '+' is not allowed near: …{s[max(0,i-6):i+8]}…")
    m = _re_space_after_minus.search(s)
    if m:
        i = m.start()
        raise ValueError(f"Space after '-' is not allowed near: …{s[max(0,i-6):i+8]}…")

    #'-' not followed by a factor (line end or SINK)
    m = _re_dangling_minus.search(s)
    if m:
        i = m.start()
        raise ValueError(f"Dangling '-' not followed by a factor near: …{s[max(0,i-6):i+8]}…")


KO = r'K\d{5}'
MOD_OR_GROUP_START = r'(?:M\d{5}|\()'
_KO_OR_MOD = re.compile(r'^(?:K\d{5}|M\d{5})$')

def normalize_line_operators(line: str) -> str:
    s = line or ""

    # 1) KO space--space KO to single space (treat as step break)
    s = re.sub(rf'(?<={KO})\s*--\s*(?={KO})', ' ', s)

    # Remove double dashes separating pathway to sink
    s = re.sub(r'\s--+\s', ' ', s)
    s = re.sub(r'\s*--+\s*', ' ', s)

    # 3) space BEFORE '-' is allowed; normalize it away if next is KO/M/(
    s = re.sub(rf'\s+-(?=(?:{KO}|{MOD_OR_GROUP_START}))', '-', s)

    # 4) collapse multi-spaces to one (keep sequences readable)
    s = re.sub(r'[ \t]+', ' ', s).strip()
    return s

def _split_top_level_by_commas(s: str):
    """Split on commas at paren-depth 0. Return list of trimmed items."""
    items, buf, depth = [], [], 0
    for ch in s:
        if ch == '(':
            depth += 1
        elif ch == ')':
            depth = max(0, depth - 1)
        if ch == ',' and depth == 0:
            item = ''.join(buf).strip()
            if item:
                items.append(item)
            buf = []
        else:
            buf.append(ch)
    tail = ''.join(buf).strip()
    if tail:
        items.append(tail)
    return items


def normalize_top_level_alternatives(line: str) -> str:
    return line


import json, numpy as np
from graphviz import Digraph


def extract_relationships(node, visited=None):
    """Turn your AST into a JSON-friendly structure that encodes probability semantics.
       Now supports nested modules via `ModuleNode` expansion.
    """
    if visited is None:
        visited = set()

    # Punctuation tokens that survive the transformer
    if isinstance(node, Token):
        if node.type in ("COMMA", "SPACE", "NEWLINE", "LPAREN",
                         "RPAREN", "PLUS", "MINUS"):
            return {"type": "empty"}
        raise TypeError(f"Unexpected bare Token in AST: {node.type}={node.value!r}")
    if isinstance(node, str):
        if not node.strip():
            return {"type": "empty"}
        raise TypeError(f"Unexpected bare str in AST: {node!r}")

    # Terminals
    if isinstance(node, KoIDNode):
        return {"type": "ko", "id": node.kid}
    if isinstance(node, OptionalNode):
        return {"type": "optional", "child": extract_relationships(node.factor, visited=visited)}
    if isinstance(node, StartNode):
        return {"type": "start"}
    if isinstance(node, SinkNode):
        return {"type": "sink"}

    # NEW: expand nested modules
    if isinstance(node, ModuleNode):
        # Inline the referenced module's relationships (already a structured subtree)
        return _module_to_relationships(node.module_id, visited=visited)

    # Binary ops
    if isinstance(node, MultiplyNode):
        if node.op == '+':
            return {
                "type": "plus",
                "rule": "multiply",
                "left": extract_relationships(node.left, visited=visited),
                "right": extract_relationships(node.right, visited=visited)
            }
        elif node.op == '-':
            return {
                "type": "minus",
                "rule": "optional_max",
                "left": extract_relationships(node.left, visited=visited),
                "right": extract_relationships(node.right, visited=visited)
            }

    # Alternatives: comma-separated
    if isinstance(node, AlternativeNode):
        return {
            "type": "alternative",
            "rule": "atleast_one",
            "branches": [extract_relationships(alt, visited=visited) for alt in node.alternatives]
        }

    # Sequence: space-separated chain
    if isinstance(node, SequenceNode):
        return {
            "type": "sequence",
            "rule": "chain_product",
            "steps": [extract_relationships(p, visited=visited) for p in node.parts]
        }

    # Program: multiple lines
    if isinstance(node, ProgramNode):
        return {
            "type": "program",
            "expressions": [extract_relationships(expr, visited=visited) for expr in node.expressions]
        }


    raise TypeError(f"Unhandled AST node in extract_relationships: {type(node).__name__}")

def load_module_expressions(input_excel: str):
    #Loads module expressions from Excel into a dict
    import pandas as pd
    df = pd.read_excel(input_excel)
    out = {}
    for _, row in df.iterrows():
        mid = str(row["Module ID"]).strip()
        expr = str(row["Module_Expression"]).strip()
        if mid and expr and expr.lower() not in ("nan", "none"):
            out[mid] = expr
    return out


def get_module_expression(module_id: str) -> str:
    #Returns the raw expression string for a KEGG module ID (e.g., 'M00010').
    global MODULE_EXPRESSION_LOOKUP
    if not MODULE_EXPRESSION_LOOKUP:
        raise RuntimeError("MODULE_EXPRESSION_LOOKUP is empty. "
                           "Call load_module_expressions(...) and assign it first.")
    expr = MODULE_EXPRESSION_LOOKUP.get(module_id)
    if not expr:
        raise KeyError(f"Module expression not found for {module_id}")
    return expr


def _module_to_relationships(module_id, visited=None):
    #Fetch a module's expression, parse it, transform to AST, then to relationships.
    if visited is None:
        visited = set()
    if module_id in visited:
        raise RuntimeError(f"Cyclic module reference: {module_id}")
    visited.add(module_id)
    try:
        expr = get_module_expression(module_id)
        txt  = preprocess_input(expr)
        # Reuse global PARSER if you made one; otherwise instantiate here:
        tree = parser.parse(txt)
        ast  = ASTTransformer().transform(tree)
        return extract_relationships(ast, visited=visited)
    finally:
        visited.remove(module_id)


import os, json
import pandas as pd
from lark import Lark

def _get_root_sequence(struct):
    if struct["type"] == "sequence":
        return struct
    if struct["type"] == "program":
        exprs = struct.get("expressions", [])
        if len(exprs) != 1 or exprs[0]["type"] != "sequence":
            raise ValueError("Expected a single top-level sequence in Program.")
        return exprs[0]
    raise ValueError(f"Root type {struct['type']} is not a sequence or program(sequence).")

def _strip_start_sink(nodes):
    return [n for n in nodes if n["type"] not in ("start", "sink")]

def top_level_steps(struct):
    seq = _get_root_sequence(struct)
    steps = _strip_start_sink(seq.get("steps", []))
    # unwrap nested single-child sequences until we reach the true step list
    while len(steps) == 1 and steps[0]["type"] == "sequence":
        steps = _strip_start_sink(steps[0].get("steps", []))
    return steps


def top_level_lines_and_steps(struct):
    """Return list of lines; each line is a list of step-structures."""
    if struct["type"] == "sequence":
        return [[s for s in struct.get("steps", []) if s["type"] not in ("start", "sink")]]
    if struct["type"] == "program":
        lines = []
        for expr in struct.get("expressions", []):
            if expr["type"] == "sequence":
                line_steps = [s for s in expr.get("steps", []) if s["type"] not in ("start", "sink")]
                lines.append(line_steps)
            else:
                lines.append([expr])
        return lines
    raise ValueError(f"Root type {struct['type']} is not a sequence or program(sequence).")


def _base_ko(ko_id: str) -> str:
    return ko_id.split("_", 1)[0] if "_" in ko_id else ko_id

def to_symbolic_plain(struct, *, drop_start_sink=True):
    t = struct["type"]

    def _wrap_if_needed(s: str) -> str:
      s = (s or "").strip()
      if not s:
          return ""
      # if already wrapped once, don't wrap again
      if s[0] == "(" and s[-1] == ")":
          return s
      # if it contains operators/spaces, wrap it
      if any(ch in s for ch in "+-* /"):
          return f"({s})"
      return s


    if t in ("start", "sink"):
        return "" if drop_start_sink else "1"
    if t == "empty":
        return ""
    if t == "ko":
        return _base_ko(struct["id"])
    if t == "optional":
        return "" #to_symbolic_plain(struct["child"], drop_start_sink=drop_start_sink)
    if t == "plus":
      L = to_symbolic_plain(struct["left"], drop_start_sink=drop_start_sink)
      R = to_symbolic_plain(struct["right"], drop_start_sink=drop_start_sink)
      if not L:
          return R or "1"
      if not R:
          return L

      L = _wrap_if_needed(L) if L else "1"
      R = _wrap_if_needed(R) if R else "1"
      return f"({L})*({R})"
    if t == "minus":
        L = to_symbolic_plain(struct["left"], drop_start_sink=drop_start_sink)
        #R = to_symbolic_plain(struct["right"], drop_start_sink=drop_start_sink)
        return L #f"max({L}, {R})"
    if t == "alternative":
        branches = [to_symbolic_plain(b, drop_start_sink=drop_start_sink) for b in struct["branches"]]
        branches = [b for b in branches if b]
        if not branches:
            return "0"
        #inner = "*".join([f"(1 - {b})" for b in branches])
        inner = "*".join([f"(1 - ({b}))" for b in branches])
        #return f"1 - ({inner})"
        return f"(1 - ({inner}))"
    if t == "sequence":
        parts = [to_symbolic_plain(x, drop_start_sink=drop_start_sink) for x in struct.get("steps", [])]
        parts = [p for p in parts if p]
        if not parts:
            return "1"
        return "*".join(_wrap_if_needed(p) for p in parts)

    if t == "program":
        exprs = [to_symbolic_plain(x, drop_start_sink=drop_start_sink)
                 for x in struct.get("expressions", [])]
        exprs = [e for e in exprs if e]
        if not exprs:
            return "1"
        if len(exprs) == 1:
            return exprs[0]
        inner = "*".join([f"(1 - ({e}))" for e in exprs])
        return f"(1 - ({inner}))"

    raise ValueError(f"Unknown type {t}")

import ast
import re
KO_RE = re.compile(r"K\d{5}")

def paren_balance(s: str) -> int:
    bal = 0
    for ch in s:
        if ch == "(":
            bal += 1
        elif ch == ")":
            bal -= 1
        if bal < 0:
            return -10**9
    return bal

def validate_equation_syntax(eq: str) -> None:
    eq = (eq or "").strip()
    if not eq:
        raise ValueError("Empty equation string")

    bal = paren_balance(eq)
    if bal != 0:
        raise ValueError(f"Unbalanced parentheses (bal={bal}) in:\n{eq}")

    # python syntax check
    try:
        ast.parse(eq, mode="eval")
    except SyntaxError as e:
        raise ValueError(f"Not valid python eval expr:\n{eq}\n{e}") from e

def eval_equation_python(eq: str, env: dict) -> float:
    """
    Evaluate with python AST but without any BLIMMP clamping.
    Only allows numbers, names, *, +, -, /, max/min calls if present.
    """
    allowed_funcs = {"max": max, "min": min}
    allowed_binops = (ast.Add, ast.Sub, ast.Mult, ast.Div)
    allowed_unary = (ast.UAdd, ast.USub)

    def _eval(node):
        if isinstance(node, ast.Expression):
            return _eval(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return float(node.value)
        if isinstance(node, ast.Name):
            # unknown KOs default to 0 unless you prefer fail-hard
            return float(env.get(node.id, 0.0))
        if isinstance(node, ast.UnaryOp) and isinstance(node.op, allowed_unary):
            v = _eval(node.operand)
            return +v if isinstance(node.op, ast.UAdd) else -v
        if isinstance(node, ast.BinOp) and isinstance(node.op, allowed_binops):
            L = _eval(node.left)
            R = _eval(node.right)
            if isinstance(node.op, ast.Add):  return L + R
            if isinstance(node.op, ast.Sub):  return L - R
            if isinstance(node.op, ast.Mult): return L * R
            if isinstance(node.op, ast.Div):  return L / R
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in allowed_funcs:
            args = [_eval(a) for a in node.args]
            return float(allowed_funcs[node.func.id](*args))
        raise ValueError(f"Unsupported AST node in generator test: {ast.dump(node)}")

    tree = ast.parse(eq, mode="eval")
    return float(_eval(tree))

def validate_extremes(eq: str) -> None:
    #TEST
    tokens = sorted(set(KO_RE.findall(eq)))

    # 0-case
    env0 = {k: 0.0 for k in tokens}
    v0 = eval_equation_python(eq, env0)

    # 1-case
    env1 = {k: 1.0 for k in tokens}
    v1 = eval_equation_python(eq, env1)

    # Range checks (most important)
    if not (0.0 <= v0 <= 1.0):
        raise ValueError(f"[range fail @0] v0={v0} eq=\n{eq}")
    if not (0.0 <= v1 <= 1.0):
        raise ValueError(f"[range fail @1] v1={v1} eq=\n{eq}")

    # “should hit extremes” only if there is at least one KO
    # (If no KOs, equation could be constant 1, etc.)
    if tokens:
        if abs(v0 - 0.0) > 1e-12:
            raise ValueError(f"[extreme fail @0] v0={v0} expected 0 eq=\n{eq}")
        if abs(v1 - 1.0) > 1e-12:
            raise ValueError(f"[extreme fail @1] v1={v1} expected 1 eq=\n{eq}")

def validate_and_report(eq: str, label: str, module_id: str, step=None, show_eq=False):
    try:
        validate_equation_syntax(eq)
        validate_extremes(eq)
        msg = f"[OK] {module_id} {label}" + (f" step={step}" if step is not None else "")
        print(msg)
        if show_eq:
            print("   ", eq)
        return True
    except Exception as e:
        msg = f"[FAIL] {module_id} {label}" + (f" step={step}" if step is not None else "")
        print(msg)
        print("   ", str(e))
        print("   eq:", eq)
        return False

FAILS = []

def build_equation_json_for_module(raw_expr: str, module_id: str):
    reset_token_counter()
    txt  = preprocess_input(raw_expr)
    print("[build] parser input lines=", txt.count("\n")+1)
    tree = Lark(grammar, parser='lalr', start='file').parse(txt)
    ast_root = ASTTransformer().transform(tree)
    rel = extract_relationships(ast_root)

    module_eq = to_symbolic_plain(rel, drop_start_sink=True)

    ok_module = validate_and_report(module_eq, "module_equation", module_id)
    if not ok_module:
        raise ValueError(f"Stopping on first failing module_equation: {module_id}")


    # Decide layout based on root kind
    if rel["type"] == "program":
        # multi-line (or single) — report per-line step equations
        lines_steps = top_level_lines_and_steps(rel)
        per_line = []
        for line_idx, step_nodes in enumerate(lines_steps, 1):
            step_eqs = []
            for i, node in enumerate(step_nodes, 1):
                eq_i = to_symbolic_plain(node, drop_start_sink=True)
                ok_step = validate_and_report(eq_i, "step_equation", module_id, step=i)
                if not ok_step:
                    #raise ValueError(f"Stopping on first failing step equation: {module_id} step {i}")
                    FAILS.append((module_id, i, eq_i))
                    continue


                step_eqs.append({"step": i, "equation": eq_i})
            per_line.append({
                "line": line_idx,
                "steps": step_eqs,
                "steps_inline": "  ".join([f"step {r['step']}: {r['equation']}" for r in step_eqs])
            })
        return {
            "module_equation": module_eq,
            "lines": per_line
        }

    # Legacy single-line (sequence) shape
    if rel["type"] == "sequence":
        steps_nodes = top_level_steps(rel)
        steps_list = []
        for i, node in enumerate(steps_nodes, 1):
            eq_i = to_symbolic_plain(node, drop_start_sink=True)

            ok_step = validate_and_report(eq_i, "step_equation", module_id, step=i)
            if not ok_step:
                #raise ValueError(f"Stopping on first failing step equation: {module_id} step {i}")
                FAILS.append((module_id, i, eq_i))
                continue


            steps_list.append({"step": i, "equation": eq_i})

        steps_inline = "  ".join([f"step {r['step']}: {r['equation']}" for r in steps_list])
        return {
            "module_equation": module_eq,
            "steps": steps_list,
            "steps_inline": steps_inline
        }

    print(FAILS)

    return {"module_equation": module_eq}



def generate_all_module_equations(input_path: str, output_dir: str,
                                  sheet=None, id_col="Module ID", expr_col="Module_Expression",
                                  consolidated_name="KEGG_Module_Equations.json",
                                  also_write_per_module=False):
    """
    Read Excel/CSV of modules, produce one consolidated JSON keyed by Module ID.
    Optionally write one JSON per module as well.
    """
    os.makedirs(output_dir, exist_ok=True)

    if input_path.lower().endswith(".csv"):
        df = pd.read_csv(input_path)
    else:
        df = pd.read_excel(input_path, sheet_name=sheet) if sheet else pd.read_excel(input_path)

    missing_cols = [c for c in (id_col, expr_col) if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    consolidated = {}
    for idx, row in df.iterrows():
        module_id = str(row[id_col]).strip()
        raw_expr  = str(row[expr_col]).strip()

        if not module_id or not raw_expr or raw_expr.lower() in ("nan", "none"):

            continue

        try:
            payload = build_equation_json_for_module(raw_expr, module_id)
            consolidated[module_id] = payload

            if also_write_per_module:
                out_path = os.path.join(output_dir, f"{module_id}_equation.json")
                with open(out_path, "w") as f:
                    json.dump({module_id: payload}, f, indent=2)
                print(f"[ok] {module_id} → {out_path}")

        except Exception as e:
            print(f"[skip] {module_id} due to error: {e}")

    consolidated_path = os.path.join(output_dir, consolidated_name)
    with open(consolidated_path, "w") as f:
        json.dump(consolidated, f, indent=2)
    print(f"\nWrote consolidated JSON: {consolidated_path}  ({len(consolidated)} modules)")

    return consolidated_path, consolidated

if __name__ == '__main__':
    input_excel = "/content/drive/MyDrive/Lab Work/Bayesian_Graph_Metabolic_Paper/KEGG_Graph_Generation/KEGG_Prokaryote_Modules_with_Definitions.xlsx"
    output_dir  = "/content/drive/MyDrive/Lab Work/Bayesian_Graph_Metabolic_Paper/BLIMMP PATH Probabilities/"
    os.makedirs(output_dir, exist_ok=True)

    MODULE_EXPRESSION_LOOKUP = load_module_expressions(input_excel)

    generate_all_module_equations(
        input_path=input_excel,
        output_dir=output_dir,
        sheet=None,
        id_col="Module ID",
        expr_col="Module_Expression",
        #consolidated_name="KEGG_Module_Equations_Jan26.json",
        consolidated_name="KEGG_Module_Equations_18AUG26.json",
        also_write_per_module=False
    )



?file: expr (NEWLINE expr)*
expr: top

# OUTSIDE parens: spaces split steps
top: step (SPACE step)*
step: choice | chain

# Steps can contain choices which are comma seperated alternatives "chains"
choice: chain (COMMA chain)+ 
# Chains can contain factors which are seperated by plus or optional nodes 
chain: factor (PLUS factor | MINUS factor)*  

# factors are units such as modules or parenthesized groups
factor: unit | group
# Parenthesized group switches to "inside parens" rules
group: LPAREN inner RPAREN

# INSIDE parens: commas = alternatives, spaces = sequence
inner: seq (COMMA seq)*             
seq: chain (SPACE chain)*           

# terminals
unit: ko_id | module | start_node | sink_node

ko_id: /-?K[0-9]{5}(?:_[0-9]+)?/
module: /M[0-9]{5}/

start_node: "START"
sink_node: "SINK"

SPACE: /[ \t]+/
COMMA: ","
PLUS: "+"
MINUS: "-"
LPAREN: "("
RPAREN: ")"
NEWLINE: "\n"

((0028)K(004B)0(0030)0(0030)8(0038)4(0034)4(0034),(002C)K(004B)1(0031)2(0032)4(0034)0(0030)7(0037),(002C)K(004B)

In [3]:
print(FAILS)

[]


In [4]:
eq = "(1 - ((1 - ((1 - ((1 - (K01243))*(1 - (K01244))))*K00899))*(1 - ((1 - ((1 - (K00772))*(1 - (K05810))))))))"

print("len:", len(eq))
print("tail:", repr(eq[-50:]))
print("balance:", paren_balance(eq))

import ast
try:
    ast.parse(eq, mode="eval")
    print("ast.parse: OK")
except SyntaxError as e:
    print("ast.parse: FAIL", e)


len: 106
tail: '))*(1 - ((1 - ((1 - (K00772))*(1 - (K05810))))))))'
balance: 0
ast.parse: OK
